# SHIPP — Run the RAG stage end to end

```text
50 listing files → 51 vision → 52 listing content → 53 search docs → 54 AI Search index → 55 eval → 56 semantic scores
```
Run `pipeline/90_run_pipeline` first (Silver listings must be current). Appends one row per stage to
`pipeline_run_log`. Set `RUN_VISION = False` to reuse existing vision results; `RUN_EVAL = False` to skip 55.
**Clear state and outputs → Run all.** Stops at the first failed gate.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
import uuid
from datetime import datetime, timezone

RUN_VISION = True
RUN_EVAL = True
STAGES = [
    ("50_silver_listing_files",   "../rag/50_silver_listing_files"),
    ("51_vision_enrichment",      "../rag/51_vision_enrichment"),
    ("52_silver_listing_content", "../rag/52_silver_listing_content"),
    ("53_gold_search_docs",       "../rag/53_gold_listing_search_docs"),
    ("54_ai_search_index",        "../rag/54_ai_search_index"),
    ("55_rag_eval",               "../rag/55_rag_retrieval_eval"),
    ("56_semantic_scores",        "../rag/56_silver_semantic_scores"),
]
if not RUN_VISION:
    STAGES = [s for s in STAGES if s[0] != "51_vision_enrichment"]
if not RUN_EVAL:
    STAGES = [s for s in STAGES if s[0] != "55_rag_eval"]

LOG_SCHEMA = ("run_id string, stage string, started_at timestamp, ended_at timestamp, "
              "seconds double, status string, summary string")
spark.sql(f"CREATE TABLE IF NOT EXISTS {PIPELINE_RUN_LOG} ({LOG_SCHEMA}) USING DELTA")
RUN_ID = str(uuid.uuid4())
print("RUN_ID =", RUN_ID)

In [0]:
log_rows, run_started = [], time.time()
try:
    for stage, path in STAGES:
        started, t0 = datetime.now(timezone.utc).replace(tzinfo=None), time.time()
        print(f"▶ {stage} ...")
        try:
            result, status = dbutils.notebook.run(path, 3600), "PASS"
        except Exception as e:
            result, status = f"{type(e).__name__}: {str(e)[:1000]}", "FAILED"
        seconds = round(time.time() - t0, 1)
        log_rows.append((RUN_ID, stage, started, datetime.now(timezone.utc).replace(tzinfo=None), seconds, status, result))
        print(f"  {status} in {seconds}s — {result}")
        if status == "FAILED":
            raise AssertionError(f"RAG pipeline stopped at {stage}. Open that notebook and run it to see the failing gate.")
finally:
    if log_rows:
        spark.createDataFrame(log_rows, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(PIPELINE_RUN_LOG)
    print(f"Total RAG pipeline time: {time.time() - run_started:.1f}s")

print("PASS — RAG pipeline completed.")
display(spark.table(PIPELINE_RUN_LOG).filter(F.col("run_id") == RUN_ID).orderBy("started_at"))